# BrainWashed model, step 1: generate training data

This notebook writes the conversations the small model learns from. Each one uses the exact prompt the BrainWashed host builds (skill index, the routed `SKILL.md` bodies, and tools), and a bigger **teacher** model writes the ideal replies.

**Before you run it**
1. Settings (right panel): **Accelerator → GPU T4 x2**, **Internet → On** (needs a phone-verified Kaggle account).
2. Pick the teacher below. The default runs **Qwen3-30B-A3B-Instruct-2507** (Apache-2.0) right here on Kaggle's two free T4s, so it costs nothing.
   To use a hosted API instead, set `USE_HOSTED_TEACHER = True` and add the key under **Add-ons → Secrets** as `TEACHER_API_KEY`. Only use a teacher whose terms allow training other models on its outputs (open-weight Qwen3 does; most closed-model APIs do not).
3. Use **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab (Kaggle allows up to 12 hours). The data ends up in this notebook's **Output**.

In [ ]:
# ---- settings ---------------------------------------------------------------
REPO = "https://github.com/ahmadalshouly/brainwashed"
BRANCH = "main"

NUM_CONVERSATIONS = 3000   # about 85-90% survive the quality filters
GEN_SKILLS = 150           # extra teacher-written skills on top of the 12 seed skills
GEN_TOOLS = 60             # extra teacher-written tools on top of the 32 seed tools
WORKERS = 8                # parallel requests to the teacher
OUT = "/kaggle/working/data"

USE_HOSTED_TEACHER = False
# Local teacher (free, runs on the Kaggle GPUs)
TEACHER_REPO = "unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF"
TEACHER_QUANT = "Q4_K_M"
LLAMA_CPP_REF = "master"   # pin a release tag like "b6500" for reproducible builds
# Hosted teacher (any OpenAI-compatible API)
HOSTED_BASE_URL = "https://openrouter.ai/api/v1"
HOSTED_MODEL = "qwen/qwen3-235b-a22b-2507"

In [ ]:
import os, subprocess, time, urllib.request

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

def build_llama_cpp(targets="llama-server llama-quantize"):
    """Builds llama.cpp with CUDA for this GPU (about 10 minutes the first time)."""
    if not os.path.exists("/tmp/llama.cpp"):
        sh(f"git clone --depth 1 --branch {LLAMA_CPP_REF} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp"
           if LLAMA_CPP_REF != "master" else "git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    arch = subprocess.check_output("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True, text=True).split()[0].replace(".", "")
    sh(f"cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={arch} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /tmp/cmake.log")
    sh(f"cmake --build /tmp/llama.cpp/build --config Release -j {os.cpu_count()} --target {targets} > /tmp/build.log")

def start_llama_server(model_path, port, extra_args="", log="/tmp/llama-server.log"):
    """Starts llama-server the way the BrainWashed host does (--jinja) and waits until it is ready."""
    cmd = (f"/tmp/llama.cpp/build/bin/llama-server -m {model_path} --jinja -ngl 999 "
           f"--host 127.0.0.1 --port {port} {extra_args}")
    print("$", cmd)
    proc = subprocess.Popen(cmd, shell=True, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(600):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2).status == 200:
                print("llama-server is ready")
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            break
        time.sleep(2)
    print(open(log).read()[-3000:])
    raise RuntimeError("llama-server did not start; see the log above")

In [ ]:
if not os.path.exists("/kaggle/working/brainwashed"):
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO} /kaggle/working/brainwashed")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Start the teacher

Skipped when `USE_HOSTED_TEACHER = True`. The model file (~18 GB) goes to `/tmp` because `/kaggle/working` only keeps 20 GB of output.

In [ ]:
if USE_HOSTED_TEACHER:
    from kaggle_secrets import UserSecretsClient
    os.environ["TEACHER_API_KEY"] = UserSecretsClient().get_secret("TEACHER_API_KEY")
    TEACHER_URL, TEACHER_MODEL = HOSTED_BASE_URL, HOSTED_MODEL
else:
    from huggingface_hub import snapshot_download
    build_llama_cpp("llama-server")
    folder = snapshot_download(TEACHER_REPO, allow_patterns=[f"*{TEACHER_QUANT}*.gguf"], local_dir="/tmp/teacher")
    gguf = sorted(str(p) for p in __import__("pathlib").Path(folder).rglob("*.gguf"))[0]
    # 8 parallel slots of 4096 tokens each, split across both T4s.
    teacher = start_llama_server(gguf, 8080, f"-c {WORKERS * 4096} -np {WORKERS}", log="/tmp/teacher.log")
    TEACHER_URL, TEACHER_MODEL = "http://127.0.0.1:8080/v1", "teacher"

## Generate

Progress prints every 25 conversations. If the run stops, run this cell again: finished conversations are kept and skipped.

In [ ]:
sh(f"cd /kaggle/working/brainwashed && python -u model/data/generate.py --base-url {TEACHER_URL} --model {TEACHER_MODEL} "
   f"--out {OUT} --num {NUM_CONVERSATIONS} --gen-skills {GEN_SKILLS} --gen-tools {GEN_TOOLS} --workers {WORKERS}")

In [ ]:
import json, collections
rows = [json.loads(l) for l in open(f"{OUT}/train.jsonl")]
rejected = [json.loads(l) for l in open(f"{OUT}/rejected.jsonl")]
print(len(rows), "training conversations,", len(rejected), "rejected")
print("most common rejection reasons:")
for reason, n in collections.Counter(r["reason"][:60] for r in rejected).most_common(5):
    print(f"  {n:>4}  {reason}")

# Look at a few examples before training on them.
for row in rows[:3]:
    print("\n" + "=" * 80, row["category"], row["skills_routed"])
    for m in row["messages"][1:]:
        body = m.get("content") or json.dumps(m.get("tool_calls"), ensure_ascii=False)
        print(f"[{m['role']}] {body[:600]}")

## Next

Click **Save Version** (if you have not already run it as a commit). Then open `02_train.ipynb`, click **Add Input → Your Work → this notebook**, and run it. The training notebook finds `train.jsonl` and `val.jsonl` automatically.